Ссылка на урок: [Спринт 2/5: 2 RAG: от сырых данных до точных ответов → Тема 3/4: Архитектура RAG → Урок 5/6: Генерация ответа и гибридный поиск]( https://practicum.yandex.ru/learn/ai-engineer/courses/99ad0dee-d9e6-4da1-b921-64e7e9653a4e/sprints/1104074/topics/b36d9f5f-3f26-4fbb-bd65-71d33c4523ef/lessons/38396bcf-f727-4087-8de4-e5b6bce19246/#133488bf-f755-4d11-bd2b-96e288a9a6be )

In [2]:
import json
from pathlib import Path

import numpy as np
from sentence_transformers import SentenceTransformer


def find_corpus() -> Path:
    """Find this notebook's corpus.json regardless of the current Jupyter working directory."""
    cwd = Path.cwd().resolve()
    candidates = []

    # Prefer this lesson's corpus.json. The repository contains other files
    # named corpus.json, for example retrieval-pipeline/corpus.json.
    for base in [cwd, *cwd.parents]:
        candidates.append(base / "response-generation-and-hybrid-search" / "corpus.json")

    # Also support running the notebook directly from its own directory.
    if cwd.name == "response-generation-and-hybrid-search":
        candidates.append(cwd / "corpus.json")

    for candidate in candidates:
        if candidate.exists():
            return candidate

    checked = "\n".join(str(p) for p in candidates[:20])
    raise FileNotFoundError(
        f"corpus.json was not found. Current directory: {cwd}\n"
        f"Checked paths:\n{checked}"
    )


corpus_path = find_corpus()
print("Using corpus:", corpus_path)

with corpus_path.open(encoding="utf-8") as f:
    data = json.load(f)
chunks = data["chunks"]        # list[dict]: 27 чанков, поля id, category, text
queries = data["queries"]      # list[dict]: 12 запросов, поля query, kind,
                               #             relevant_ids, gold
texts = [c["text"] for c in chunks]

print("Чанков:", len(chunks), "| Запросов:", len(queries))
print("Смысловой запрос:", queries[1])
print("Запрос с точным термином:", queries[7])

# Вывод:
# Чанков: 27 | Запросов: 12
# Смысловой запрос: {'query': 'Что предпринять, если устройство не выходит
#   в интернет?', 'kind': 'semantic', 'relevant_ids': [15], 'gold': 'провод оператора'}
# Запрос с точным термином: {'query': 'что означает код ошибки E-52',
#   'kind': 'exact', 'relevant_ids': [4], 'gold': 'нагреве процессора'}

# --- Эмбеддер: тот же multilingual-e5, что и в теме (на GPU) ---
embedder = SentenceTransformer("intfloat/multilingual-e5-base", device="cuda")

# Документы кодируем заранее с префиксом passage: — предвычисление индекса
doc_vecs = embedder.encode(["passage: " + t for t in texts],
                           normalize_embeddings=True, show_progress_bar=False)


def dense_search(query, k=3):
    """Плотный (векторный) поиск: индексы top-k чанков по косинусному
    сходству. query кодируется с префиксом query:, как во всей теме."""
    q_vec = embedder.encode("query: " + query, normalize_embeddings=True)
    order = np.argsort(q_vec @ doc_vecs.T)[::-1][:k]
    return [int(i) for i in order]                    # Индексы чанков как int

Using corpus: C:\PythonProg\all-agent.rag-arch\response-generation-and-hybrid-search\corpus.json
Чанков: 27 | Запросов: 12
Смысловой запрос: {'query': 'Что предпринять, если устройство не выходит в интернет?', 'kind': 'semantic', 'relevant_ids': [15], 'gold': 'провод оператора'}
Запрос с точным термином: {'query': 'что означает код ошибки E-52', 'kind': 'exact', 'relevant_ids': [4], 'gold': 'нагреве процессора'}
